# Test
This is a test project to be used in presentations. Let's check using public data.
Data link : https://www.kaggle.com/datasets/tunguz/online-retail-ii

Let's apply my notebook notes over this data.

In the first stage, let's fetch the data from Kaggle, clean it, and apply necessary EDA processes.

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tunguz/online-retail-ii")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'online-retail-ii' dataset.
Path to dataset files: /kaggle/input/online-retail-ii


In [ ]:
import pandas as pd

# Data Import
df = pd.read_csv(path + "/online_retail_II.csv", encoding="latin1")
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850.0,United Kingdom


In [ ]:
# For data inspection
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      541910 non-null  object 
 1   StockCode    541910 non-null  object 
 2   Description  540456 non-null  object 
 3   Quantity     541910 non-null  int64  
 4   InvoiceDate  541910 non-null  object 
 5   Price        541910 non-null  float64
 6   Customer ID  406830 non-null  float64
 7   Country      541910 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


# Data Info Summary
datetime is an object --> we must parse this into date correctly.

total_paid column can be created --> Quantity * Price

Country and Description columns can be dropped

In [ ]:
# Number of duplicate rows in data
df.duplicated().sum()

np.int64(0)

In [ ]:
# Cleaning duplicate data
df.drop_duplicates(inplace=True, ignore_index=True, keep="first")
df.duplicated().sum()

np.int64(0)

In [ ]:
# Distribution of numerical columns in the data
df.describe()

,Quantity,Price,Customer ID
count,536642.000000,536642.000000,401605.000000
mean,9.620013,4.632681,15281.154341
std,219.129952,97.233029,1714.008869
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13939.000000
50%,3.000000,2.080000,15145.000000
75%,10.000000,4.130000,16784.000000
max,80995.000000,38970.000000,18287.000000


In [ ]:
# Check for the distribution of negative '-' values
df[df['Price']<=0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
605,536414,22139,NaN,56,12/1/10 11:52,0.0,NaN,United Kingdom
1474,536545,21134,NaN,1,12/1/10 14:32,0.0,NaN,United Kingdom
1949,536547,37509,NaN,1,12/1/10 14:33,0.0,NaN,United Kingdom
1950,536546,22145,NaN,1,12/1/10 14:33,0.0,NaN,United Kingdom
1982,536552,20950,NaN,1,12/1/10 14:34,0.0,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
531753,581234,72817,NaN,27,12/8/11 10:33,0.0,NaN,United Kingdom
533259,581406,46000M,POLYESTER FILLER PAD 45x45cm,240,12/8/11 13:58,0.0,NaN,United Kingdom
533260,581406,46000S,POLYESTER FILLER PAD 40x40cm,300,12/8/11 13:58,0.0,NaN,United Kingdom
533309,581408,85175,NaN,20,12/8/11 14:06,0.0,NaN,United Kingdom


There are negative "-" values in quantity and Price columns. Since it's a test project, these are useless to me. My goal here is segmentation and churn prediction.

In [ ]:
# I am dropping Customer ID NaN rows since I need valid IDs for segmentation
df.dropna(subset=['Customer ID'], inplace = True)

# Let's clean the negative (-) values
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

# Check after cleaning
df.describe()

,Quantity,Price,Customer ID
count,392693.000000,392693.000000,392693.000000
mean,13.119671,3.125952,15287.837224
std,180.492603,22.241820,1713.542421
min,1.000000,0.001000,12346.000000
25%,2.000000,1.250000,13955.000000
50%,6.000000,1.950000,15150.000000
75%,12.000000,3.750000,16791.000000
max,80995.000000,8142.750000,18287.000000


In [ ]:
# Number of rows left after cleanup
print(len(df))

392693


In [ ]:
# Fixing date format is critical for 'Sliding Window'
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Calculate total revenue (Quantity * Price)
df['TotalRevenue'] = df['Quantity'] * df['Price']

/tmp/ipython-input-477170097.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])


In [ ]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalRevenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [ ]:
# Any missing rows left in data?
df.isnull().sum()

,0
Invoice,0
StockCode,0
Description,0
Quantity,0
InvoiceDate,0
Price,0
Customer ID,0
Country,0
TotalRevenue,0


# Data Cleaning
Right now, the data looks clean for my use case. Let's build the Sliding Window logic for the project.

In [ ]:
# Last transaction date in data
today_date = df['InvoiceDate'].max()
print(f"Verideki en güncel tarih: {today_date}")

# Set analysis date as 2 days after the last date (standard practice)
import datetime as dt
analysis_date = today_date + dt.timedelta(days=2)

Verideki en güncel tarih: 2011-12-09 12:50:00


In [ ]:
# Customer based grouping
rfm = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda date: (analysis_date - date.max()).days, # Recency
    'Invoice': lambda num: num.nunique(),                         # Frequency
    'TotalRevenue': lambda price: price.sum()                     # Monetary
})

rfm.columns = ['recency', 'frequency', 'monetary']

In [ ]:
# Separating last 30 days vs the rest of the timelines
cutoff_date = analysis_date - dt.timedelta(days=30)
active_window = df[df['InvoiceDate'] >= cutoff_date]
past_window = df[df['InvoiceDate'] < cutoff_date]

In [ ]:
# What transactions did customers perform in these intervals
monetary_active = active_window.groupby('Customer ID')['TotalRevenue'].sum()
monetary_past = past_window.groupby('Customer ID')['TotalRevenue'].sum()
df_merge = pd.merge(monetary_active, monetary_past, on='Customer ID', how='outer', suffixes=('_active', '_past'))
df_merge.fillna(0, inplace=True)

# What did we do?
Since we need to proceed over dates for Slide Window logic, we created a pseudo-today point and added 2 days buffer for recency.

We aggregated tracking for RFM base: Customer-based Recency, Frequency, and Monetary values.

Making a change to the plan, I performed 2 operations: inspecting the last 30 days vs the last 1-year data. The reason is to discover inactive members who made very high spendings in the past.

We merge the sales and data obtained from these 2 datasets. If there are customers missing during outer join, we fill NaN values with 0.

In [ ]:
df_merge.head()

,TotalRevenue_active,TotalRevenue_past
Customer ID,,
12346.0,0.00,77183.60
12347.0,224.82,4085.18
12348.0,0.00,1797.24
12349.0,1757.55,0.00
12350.0,0.00,334.40


In [ ]:
# Let's determine the limits based on the data. However, we could also supply these manually.
df_merge['TotalRevenue_active'].quantile([0.25, 0.50, 0.75, 0.90, 0.95])

,TotalRevenue_active
0.25,0.0000
0.50,0.0000
0.75,231.9875
0.90,597.0780
0.95,980.3665


In [ ]:
# Segmentation function
def customer_segment(df):
  if df['TotalRevenue_active'] > 0 and df['TotalRevenue_active'] <=250:
    return 'Bronze'
  elif df['TotalRevenue_active'] > 250 and df['TotalRevenue_active'] <= 600 :
    return 'Silver'
  elif df['TotalRevenue_active'] > 600 and df['TotalRevenue_active'] <= 1000:
    return 'Gold'
  elif df['TotalRevenue_active'] > 1000:
    return 'Diamond'
  elif df['TotalRevenue_active'] == 0 and df['TotalRevenue_past'] > 1000:
    return 'Passive Premium'
  else:
    return 'Other'

In [ ]:
df_merge['Segment'] = df_merge.apply(customer_segment, axis=1)

I wanted to apply percentile segmentation on our merge table based on quantiles. I mapped these into groups. Now every customer has a segment, we can jump to ML to cluster them.

Another differentiator for KMeans is Frequency, which helps distinguish between customers spending 1000 TL once vs 100 TL spread over 10 times.

In [ ]:
# The spendings and segments are clear, but for KMeans, I also need frequency of actions.
df_merge['Frequency_active'] = active_window.groupby('Customer ID')['Invoice'].nunique()
df_merge['Frequency_past'] = past_window.groupby('Customer ID')['Invoice'].nunique()
df_merge.fillna(0, inplace=True)
df_merge.head()

,TotalRevenue_active,TotalRevenue_past,Segment,Frequency_active,Frequency_past
Customer ID,,,,,
12346.0,0.00,77183.60,Passive Premium,0.0,1.0
12347.0,224.82,4085.18,Bronze,1.0,6.0
12348.0,0.00,1797.24,Passive Premium,0.0,4.0
12349.0,1757.55,0.00,Diamond,1.0,0.0
12350.0,0.00,334.40,Other,0.0,1.0


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [ ]:
# Select numerical columns which will go into model
modele_girecekler = ['Frequency_active', 'TotalRevenue_active']

for segment_ismi in df_merge['Segment'].unique():
    # Filter over original text
    segment_verisi = df_merge[df_merge['Segment'] == segment_ismi].copy()

    # Scale only numerical columns
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(segment_verisi[modele_girecekler])

    # Run KMeans with this clean data
    kmeans = KMeans(n_clusters=2)
    segment_verisi['Cluster'] = kmeans.fit_predict(X_scaled)

    # Project results back onto main table
    # (Index-based mapping is the safest method here)
    df_merge.loc[segment_verisi.index, 'Cluster'] = segment_verisi['Cluster']

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (2). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (2). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


In [ ]:
df_merge['Sub_Segment'] = 'Normal' # All members default to Normal subsegment.

for segment_name in df_merge['Segment'].unique():
    segment_data = df_merge[df_merge['Segment'] == segment_name]

    # Checking for sufficient data size
    if len(segment_data['Cluster'].unique()) > 1:
        cluster_0_freq = segment_data[segment_data['Cluster'] == 0]['Frequency_active'].mean()
        cluster_1_freq = segment_data[segment_data['Cluster'] == 1]['Frequency_active'].mean()

        # We take the mean of each segment's active column. If individuals have higher frequency, they get upgraded to Plus.
        if cluster_0_freq > cluster_1_freq:
            df_merge.loc[(df_merge['Segment'] == segment_name) & (df_merge['Cluster'] == 0), 'Sub_Segment'] = segment_name + ' Plus'
            df_merge.loc[(df_merge['Segment'] == segment_name) & (df_merge['Cluster'] == 1), 'Sub_Segment'] = segment_name + ' Normal'
        else:
            df_merge.loc[(df_merge['Segment'] == segment_name) & (df_merge['Cluster'] == 1), 'Sub_Segment'] = segment_name + ' Plus'
            df_merge.loc[(df_merge['Segment'] == segment_name) & (df_merge['Cluster'] == 0), 'Sub_Segment'] = segment_name + ' Normal'
    else:
        # If there's no comparable cluster in the data, it stays 'Normal'.
        df_merge.loc[(df_merge['Segment'] == segment_name), 'Sub_Segment'] = segment_name + ' Normal'

print("Sub-Segment Value Counts:")
print(df_merge['Sub_Segment'].value_counts())

print("\nSample Data with new Sub_Segment:")
print(df_merge[['Segment', 'Cluster', 'Sub_Segment']].head())

Sub-Segment Value Counts:
Sub_Segment
Other Normal              2050
Passive Premium Normal     737
Bronze Normal              473
Silver Normal              448
Diamond Plus               207
Silver Plus                167
Gold Normal                144
Gold Plus                   80
Bronze Plus                 31
Diamond Normal               1
Name: count, dtype: int64

Sample Data with new Sub_Segment:
                     Segment  Cluster             Sub_Segment
Customer ID                                                  
12346.0      Passive Premium      0.0  Passive Premium Normal
12347.0               Bronze      1.0           Bronze Normal
12348.0      Passive Premium      0.0  Passive Premium Normal
12349.0              Diamond      0.0            Diamond Plus
12350.0                Other      0.0            Other Normal


# XGBoost Architecture
We have completed the KMeans part so far.

I did manual classification, added frequency to data, and clustered it using kmeans. So even if 0 and 1 are irrelevant for this algorithm, I used mean aggregation to classify Premium vs Normal customers.

Now I will add some more EDA for the XGBoost algorithm (Churn probability) and perform model training.


In [ ]:
# Preparing the data for XGBoost algorithm, namely for predicting churn probabilities
# To avoid infinity division by 0 in TotalRev columns, we add 1 to prevent data corruption.
df_merge['spending_momentum'] = df_merge['TotalRevenue_active'] / (df_merge['TotalRevenue_past']+1)
df_merge['frequency_ratio'] = df_merge['Frequency_active'] / (df_merge['Frequency_past']+1)
df_merge.head()

,TotalRevenue_active,TotalRevenue_past,Segment,Frequency_active,Frequency_past,Cluster,Sub_Segment,spending_momentum,frequency_ratio
Customer ID,,,,,,,,,
12346.0,0.00,77183.60,Passive Premium,0.0,1.0,0.0,Passive Premium Normal,0.00000,0.000000
12347.0,224.82,4085.18,Bronze,1.0,6.0,1.0,Bronze Normal,0.05502,0.142857
12348.0,0.00,1797.24,Passive Premium,0.0,4.0,0.0,Passive Premium Normal,0.00000,0.000000
12349.0,1757.55,0.00,Diamond,1.0,0.0,0.0,Diamond Plus,1757.55000,1.000000
12350.0,0.00,334.40,Other,0.0,1.0,0.0,Other Normal,0.00000,0.000000


In [ ]:
# Customers with 0 spending in last 30 days are Churn (1), active are (0)
df_merge['is_churn'] = (df_merge['TotalRevenue_active'] == 0).astype(int)

# Let's check the churn rate
print(df_merge['is_churn'].value_counts(normalize=True))

is_churn
1    0.642462
0    0.357538
Name: proportion, dtype: float64


In [ ]:
# Extracting X and y features
X = df_merge[['TotalRevenue_past', 'Frequency_past']]
y = df_merge['is_churn']

In [ ]:
# Imports
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score, confusion_matrix

In [ ]:
# Data splitting (to prevent Data Leakage)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Balancing Class Destribution using SMOTE
# Note: This block was added for analysis purposes, not run yet.
# Required Library: pip install imbalanced-learn
from imblearn.over_sampling import SMOTE
import pandas as pd

print("SMOTE Öncesi Eğitim Sınıf Dağılımı:\n", pd.Series(y_train).value_counts())

# Create SMOTE Object
smote = SMOTE(random_state=42)

# Balance the data (X_train, y_train)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("SMOTE Sonrası Sınıf Dağılımı:\n", pd.Series(y_train_smote).value_counts())

# You can train using balanced resampled data:
xgb_smote = XGBClassifier(max_depth=5, learning_rate=0.1, n_estimators=50)
xgb_smote.fit(X_train_smote, y_train_smote)


In [ ]:
# First model training
xgb = XGBClassifier(
    max_depth=7,
    learning_rate=0.2,
    n_estimators=100,
)
xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.2, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
# Evaluation function
def test_model(model, X_train, y_train, X_test, y_test):
  y_train_pred = model.predict(X_train)
  y_test_pred = model.predict(X_test)

  print(f"Train Accuracy: {accuracy_score(y_train, y_train_pred)}")
  print(f"Test Accuracy: {accuracy_score(y_test, y_test_pred)}")
  print("\nTest Classification Report:")
  print(classification_report(y_test, y_test_pred))
  print("\nTest Confusion Matrix:")
  print(confusion_matrix(y_test, y_test_pred))

In [ ]:
test_model(xgb, X_train, y_train, X_test, y_test)

Train Accuracy: 0.7979827089337176
Test Accuracy: 0.7269585253456221

Test Classification Report:
              precision    recall  f1-score   support

           0       0.68      0.45      0.54       310
           1       0.74      0.88      0.81       558

    accuracy                           0.73       868
   macro avg       0.71      0.66      0.67       868
weighted avg       0.72      0.73      0.71       868


Test Confusion Matrix:
[[138 172]
 [ 65 493]]


In [ ]:
# Let's check with grid search to improve hyperparameters in addition to the first random model.
from sklearn.model_selection import GridSearchCV

In [ ]:
grid = GridSearchCV(
    estimator=xgb,
    param_grid={
        'max_depth': [3, 4, 5, 7],
        'learning_rate': [0.1, 0.2, 0.3],
        'n_estimators': [50, 100, 200]
    },
)
grid.fit(X_train, y_train)

GridSearchCV(estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False, eval_metric=None,
                                     feature_types=None, feature_weights=None,
                                     gamma=None, grow_policy=None,
                                     importance_type=None,
                                     interaction_constraints=None,
                                     learning_rate=0.2, max_bin=None,
                                     max_cat_threshold=None,
                                     max_cat_to_onehot=None,
                                     max_delta_step=None, max_depth=7,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=100,
                                     n_jobs=None, num_parallel_tree=None, ...),
             param_grid={'learning_rate': [0.1, 0.2, 0.3],
                         'max_depth': [3, 4, 5, 7],
                         'n_estimators': [50, 100, 200]})

In [ ]:
# Best hyperparameters found
grid.best_params_

{'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50}

In [ ]:
# Training new model with the best parameters found
xgb_1 = XGBClassifier(
    max_depth=5,
    learning_rate=0.1,
    n_estimators=50,
)
xgb_1.fit(X_train, y_train)
#

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=50,
              n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
test_model(xgb_1,  X_train, y_train, X_test, y_test)

Train Accuracy: 0.7685878962536024
Test Accuracy: 0.7407834101382489

Test Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.45      0.55       310
           1       0.75      0.91      0.82       558

    accuracy                           0.74       868
   macro avg       0.73      0.68      0.68       868
weighted avg       0.74      0.74      0.72       868


Test Confusion Matrix:
[[138 172]
 [ 53 505]]


In [ ]:
# Appending churn probabilities to data as a column.
df_merge['Churn Proba'] = xgb_1.predict_proba(df_merge[['TotalRevenue_past', 'Frequency_past']])[:, 1]

In [ ]:
df_merge.head()

,TotalRevenue_active,TotalRevenue_past,Segment,Frequency_active,Frequency_past,...,Sub_Segment,spending_momentum,frequency_ratio,is_churn,Churn Proba
Customer ID,,,,,,,,,,,
12346.0,0.00,77183.60,Passive Premium,0.0,1.0,...,Passive Premium Normal,0.00000,0.000000,1,0.939889
12347.0,224.82,4085.18,Bronze,1.0,6.0,...,Bronze Normal,0.05502,0.142857,0,0.320716
12348.0,0.00,1797.24,Passive Premium,0.0,4.0,...,Passive Premium Normal,0.00000,0.000000,1,0.584760
12349.0,1757.55,0.00,Diamond,1.0,0.0,...,Diamond Plus,1757.55000,1.000000,0,0.008074
12350.0,0.00,334.40,Other,0.0,1.0,...,Other Normal,0.00000,0.000000,1,0.818388


### Conclusion and Evaluation
The main goal of this model is to **accurately predict the probability of customers churning.** Our model reached an impressive 91% recall score for detecting churning customers (Class 1).

However, due to *class imbalance* detected in our dataset, our success in predicting active/non-churning customers (Class 0) remained relatively lower. To resolve this, **SMOTE** synthetic data generation was tested, but no significant improvement was observed.

In conclusion, instead of artificially upsampling the dataset, it will be much healthier to resolve this imbalance issue by **feeding the algorithm with more real customer data (increasing learning samples)**.